In [3]:
import pandas as pd

In [5]:

def convert_to_pandas(dataset, batch_size=1000):
    """
    Convert a Hugging Face dataset (streaming or not) to a pandas DataFrame in batches.
    """
    df_list = []
    
    i=0
    # Hugging Face streaming datasets use iter() for batching
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch {i} \n")
        i += 1
        # batch is a dict, convert directly to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

def sample_hf_data(data, sample_size=1000):

    # Shuffle the dataset (buffer_size controls memory usage)
    shuffled = data.shuffle(buffer_size=sample_size)
    # Take random samples
    sampled_dataset = shuffled.take(sample_size)
    
    return sampled_dataset

In [17]:
"""
Collect evaluation datasets across all competition domains
Creates a unified JSON file with question-answer pairs for evaluation
"""

from datasets import load_dataset
import json
from typing import List, Dict
import random

# Set seed for reproducibility
random.seed(42)

def sample_hf_streaming(dataset_name: str, config: str, split: str, n_samples: int, buffer_size: int = 10000):
    """Sample from HuggingFace dataset using streaming for fast loading"""
    print(f"    Streaming {n_samples} samples from {dataset_name}...", end=" ", flush=True)
    dataset = load_dataset(dataset_name, config, split=split, streaming=True)
    shuffled = dataset.shuffle(buffer_size=buffer_size, seed=42)
    
    # Collect samples with progress
    samples = []
    for i, item in enumerate(shuffled):
        samples.append(item)
        if (i + 1) % 10 == 0:
            print(f"{i+1}", end="...", flush=True)
        if len(samples) >= n_samples:
            break
    
    print(f" Done!")
    return samples

def sample_dataset(dataset, n_samples: int, question_col: str, answer_col: str, domain: str) -> List[Dict]:
    """Sample n examples from a dataset (for non-streaming datasets)"""
    print(f"    Loading dataset...", end=" ", flush=True)
    samples = []
    
    # Convert to list and sample
    data_list = list(dataset)
    print(f"{len(data_list)} total examples.", end=" ", flush=True)
    
    if len(data_list) > n_samples:
        data_list = random.sample(data_list, n_samples)
    
    print(f"Sampling {len(data_list)} examples...", end=" ", flush=True)
    
    for item in data_list:
        samples.append({
            "domain": domain,
            "question": item[question_col],
            "answer": item[answer_col],
            "source": domain
        })
    
    print("Done!")
    return samples

def sample_dataset_fast(dataset_name: str, config: str, split: str, n_samples: int, 
                       question_col: str, answer_col: str, domain: str, 
                       transform_fn=None) -> List[Dict]:
    """Fast sampling using streaming"""
    try:
        items = sample_hf_streaming(dataset_name, config, split, n_samples)
        samples = []
        for item in items:
            sample = {
                "domain": domain,
                "question": item[question_col],
                "answer": item[answer_col] if not transform_fn else transform_fn(item[answer_col]),
                "source": domain
            }
            samples.extend([sample] if not isinstance(sample, list) else sample)
        return samples
    except Exception as e:
        print(f"  ✗ Streaming failed for {domain}: {e}")
        return []

def collect_datasets(samples_per_domain: int = 50):
    """Collect evaluation datasets across all domains"""
    
    all_samples = []
    total_domains = 17
    current = 0
    
    print("="*60)
    print("COLLECTING EVALUATION DATASETS")
    print("="*60)
    
    # 1. MATH REASONING
    current += 1
    print(f"\n[{current}/{total_domains}] MATH REASONING")
    print("-" * 60)
    
    # GSM8K
    try:
        gsm8k = load_dataset("openai/gsm8k", "main", split="test")
        samples = sample_dataset(gsm8k, samples_per_domain, "question", "answer", "math_gsm8k")
        # Extract numeric answer from #### format
        for s in samples:
            if "####" in s["answer"]:
                s["answer"] = s["answer"].split("####")[1].strip()
        all_samples.extend(samples)
        print(f"  ✓ GSM8K: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ GSM8K failed: {e}")
    
    # MATH dataset
    try:
        math_dataset = load_dataset("hendrycks/math", split="test", streaming=True)
        items = list(math_dataset.shuffle(buffer_size=5000, seed=42).take(samples_per_domain))
        samples = []
        for item in items:
            samples.append({
                "domain": "math_competition",
                "question": item["problem"],
                "answer": item["solution"],
                "source": "math_competition"
            })
        all_samples.extend(samples)
        print(f"  ✓ MATH: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ MATH failed: {e}")
    
    # 2. CODE GENERATION
    current += 1
    print(f"\n[{current}/{total_domains}] CODE GENERATION")
    print("-" * 60)
    
    # HumanEval
    try:
        humaneval = load_dataset("openai/openai_humaneval", split="test")
        samples = []
        for item in list(humaneval)[:samples_per_domain]:
            samples.append({
                "domain": "code_humaneval",
                "question": item["prompt"],
                "answer": item["canonical_solution"],
                "source": "code_humaneval"
            })
        all_samples.extend(samples)
        print(f"  ✓ HumanEval: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ HumanEval failed: {e}")
    
    # MBPP
    try:
        mbpp = load_dataset("google-research-datasets/mbpp", "sanitized", split="test")
        print(f"    Loading MBPP...", end=" ", flush=True)
        samples = []
        for item in list(mbpp)[:samples_per_domain]:
            samples.append({
                "domain": "code_mbpp",
                "question": item["prompt"],  # Changed from 'text' to 'prompt'
                "answer": item["code"],
                "source": "code_mbpp"
            })
        all_samples.extend(samples)
        print(f"Done! {len(samples)} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # 3. SCIENCE REASONING
    current += 1
    print(f"\n[{current}/{total_domains}] SCIENCE REASONING")
    print("-" * 60)
    
    # GPQA Diamond
    try:
        gpqa = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
        print(f"    Loading GPQA Diamond...", end=" ", flush=True)
        samples = []
        for item in list(gpqa)[:samples_per_domain]:
            # Format multiple choice question
            question = f"{item['Question']}\nA) {item['Incorrect Answer 1']}\nB) {item['Incorrect Answer 2']}\nC) {item['Incorrect Answer 3']}\nD) {item['Correct Answer']}"
            samples.append({
                "domain": "science_gpqa",
                "question": question,
                "answer": "D",  # Correct answer is always D in this format
                "source": "science_gpqa"
            })
        all_samples.extend(samples)
        print(f"Done! {len(samples)} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # ARC Challenge
    try:
        arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
        print(f"    Loading ARC Challenge...", end=" ", flush=True)
        samples = []
        for item in list(arc)[:samples_per_domain]:
            # Format question with choices
            question = item['question']
            choices = item['choices']
            question += "\n" + "\n".join([f"{choices['label'][i]}) {choices['text'][i]}" 
                                         for i in range(len(choices['label']))])
            samples.append({
                "domain": "science_arc",
                "question": question,
                "answer": item['answerKey'],
                "source": "science_arc"
            })
        all_samples.extend(samples)
        print(f"Done! {len(samples)} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # 4. FACTUAL QA
    current += 1
    print(f"\n[{current}/{total_domains}] FACTUAL QA")
    print("-" * 60)
    
    # TriviaQA - using streaming for speed
    try:
        items = sample_hf_streaming("mandarjoshi/trivia_qa", "rc.nocontext", "validation", samples_per_domain)
        samples = []
        for item in items:
            samples.append({
                "domain": "qa_trivia",
                "question": item["question"],
                "answer": item["answer"]["value"],
                "source": "qa_trivia"
            })
        all_samples.extend(samples)
        print(f"  ✓ TriviaQA: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ TriviaQA failed: {e}")
    
    # Natural Questions - using streaming
    try:
        items = sample_hf_streaming("google-research-datasets/natural_questions", None, "validation", samples_per_domain * 3, buffer_size=5000)
        samples = []
        for item in items:
            # Only use examples with short answers
            try:
                if item['annotations']['short_answers'] and len(item['annotations']['short_answers']) > 0:
                    short_ans = item['annotations']['short_answers'][0]
                    if isinstance(short_ans, list) and len(short_ans) > 0:
                        answer_text = short_ans[0].get('text', '')
                        if answer_text:
                            samples.append({
                                "domain": "qa_natural",
                                "question": item["question"]["text"],
                                "answer": answer_text,
                                "source": "qa_natural"
                            })
            except (KeyError, IndexError, TypeError):
                continue
            
            if len(samples) >= samples_per_domain:
                break
        all_samples.extend(samples[:samples_per_domain])
        print(f"  ✓ Natural Questions: {len(samples[:samples_per_domain])} samples")
    except Exception as e:
        print(f"  ✗ Natural Questions failed: {e}")
    
    # 5. READING COMPREHENSION
    current += 1
    print(f"\n[{current}/{total_domains}] READING COMPREHENSION")
    print("-" * 60)
    
    # DROP - using streaming
    try:
        items = sample_hf_streaming("ucinlp/drop", None, "validation", samples_per_domain)
        samples = []
        for item in items:
            samples.append({
                "domain": "rc_drop",
                "question": f"Context: {item['passage']}\n\nQuestion: {item['question']}",
                "answer": item["answers_spans"]["spans"][0] if item["answers_spans"]["spans"] else str(item["answers_spans"]["number"]),
                "source": "rc_drop"
            })
        all_samples.extend(samples)
        print(f"  ✓ DROP: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ DROP failed: {e}")
    
    # 6. COMMON SENSE REASONING
    current += 1
    print(f"\n[{current}/{total_domains}] COMMON SENSE REASONING")
    print("-" * 60)
    
    # CommonsenseQA
    try:
        csqa = load_dataset("tau/commonsense_qa", split="validation")
        samples = []
        for item in list(csqa)[:samples_per_domain]:
            question = item['question']
            choices = item['choices']
            question += "\n" + "\n".join([f"{choices['label'][i]}) {choices['text'][i]}" 
                                         for i in range(len(choices['label']))])
            samples.append({
                "domain": "commonsense_qa",
                "question": question,
                "answer": item['answerKey'],
                "source": "commonsense_qa"
            })
        all_samples.extend(samples)
        print(f"  ✓ CommonsenseQA: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ CommonsenseQA failed: {e}")
    
    # HellaSwag - SKIPPED due to disk space
    print("  ⊘ HellaSwag: Skipped (disk space)")
    
    # 7. SUMMARIZATION
    current += 1
    print(f"\n[{current}/{total_domains}] SUMMARIZATION")
    print("-" * 60)
    
    # CNN/DailyMail - using streaming
    try:
        items = sample_hf_streaming("abisee/cnn_dailymail", "3.0.0", "test", samples_per_domain)
        samples = []
        for item in items:
            samples.append({
                "domain": "summarization_cnn",
                "question": item["article"],
                "answer": item["highlights"],
                "source": "summarization_cnn"
            })
        all_samples.extend(samples)
        print(f"  ✓ CNN/DailyMail: {len(samples)} samples")
    except Exception as e:
        print(f"  ✗ CNN/DailyMail failed: {e}")
    
    # XSum - SKIPPED due to dataset script issue
    print("  ⊘ XSum: Skipped (dataset deprecated)")
    
    # 8. MULTIPLE CHOICE KNOWLEDGE
    current += 1
    print(f"\n[{current}/{total_domains}] KNOWLEDGE (MMLU)")
    print("-" * 60)
    
    # MMLU - SKIPPED due to disk space
    print("  ⊘ MMLU: Skipped (disk space)")
    
    # 9. CREATIVE WRITING (using prompts)
    current += 1
    print(f"\n[{current}/{total_domains}] CREATIVE WRITING")
    print("-" * 60)
    print(f"    Adding creative writing prompts...", end=" ", flush=True)
    creative_prompts = [
        {"domain": "creative_writing", "question": "Write a short story about a time traveler who discovers they can't change the past.", "answer": "[Creative writing - no single answer]", "source": "creative_writing"},
        {"domain": "creative_writing", "question": "Write a poem about the feeling of nostalgia on a rainy day.", "answer": "[Creative writing - no single answer]", "source": "creative_writing"},
        {"domain": "creative_writing", "question": "Create a dialogue between two characters meeting for the first time at a coffee shop.", "answer": "[Creative writing - no single answer]", "source": "creative_writing"},
        {"domain": "creative_writing", "question": "Write a descriptive paragraph about a futuristic city.", "answer": "[Creative writing - no single answer]", "source": "creative_writing"},
        {"domain": "creative_writing", "question": "Create a character sketch of an eccentric inventor.", "answer": "[Creative writing - no single answer]", "source": "creative_writing"},
    ] * 10  # Duplicate to get 50 samples
    all_samples.extend(creative_prompts[:samples_per_domain])
    print(f"Done! {samples_per_domain} samples")
    
    # 10. CREATIVE IDEATION
    current += 1
    print(f"\n[{current}/{total_domains}] CREATIVE IDEATION")
    print("-" * 60)
    print(f"    Adding ideation prompts...", end=" ", flush=True)
    ideation_prompts = [
        {"domain": "creative_ideation", "question": "Brainstorm 5 innovative uses for recycled plastic bottles.", "answer": "[Ideation - multiple valid answers]", "source": "creative_ideation"},
        {"domain": "creative_ideation", "question": "Generate creative names for a tech startup focused on sustainable energy.", "answer": "[Ideation - multiple valid answers]", "source": "creative_ideation"},
        {"domain": "creative_ideation", "question": "List unique marketing strategies for a local bookstore.", "answer": "[Ideation - multiple valid answers]", "source": "creative_ideation"},
        {"domain": "creative_ideation", "question": "What are some creative solutions to reduce food waste in restaurants?", "answer": "[Ideation - multiple valid answers]", "source": "creative_ideation"},
        {"domain": "creative_ideation", "question": "Suggest innovative features for a next-generation smartwatch.", "answer": "[Ideation - multiple valid answers]", "source": "creative_ideation"},
    ] * 10
    all_samples.extend(ideation_prompts[:samples_per_domain])
    print(f"Done! {samples_per_domain} samples")
    
    # 11. CONVERSATIONAL/ADVICE
    current += 1
    print(f"\n[{current}/{total_domains}] CONVERSATION/ADVICE")
    print("-" * 60)
    print(f"    Adding conversation prompts...", end=" ", flush=True)
    conversation_prompts = [
        {"domain": "conversation", "question": "I'm feeling overwhelmed with work deadlines. What advice do you have?", "answer": "[Conversational - subjective]", "source": "conversation"},
        {"domain": "conversation", "question": "How should I prepare for my first job interview?", "answer": "[Conversational - subjective]", "source": "conversation"},
        {"domain": "conversation", "question": "What are some good habits for maintaining work-life balance?", "answer": "[Conversational - subjective]", "source": "conversation"},
        {"domain": "conversation", "question": "I want to learn a new programming language. Where should I start?", "answer": "[Conversational - subjective]", "source": "conversation"},
        {"domain": "conversation", "question": "How can I improve my public speaking skills?", "answer": "[Conversational - subjective]", "source": "conversation"},
    ] * 10
    all_samples.extend(conversation_prompts[:samples_per_domain])
    print(f"Done! {samples_per_domain} samples")
    
    # Summary
    print("\n" + "="*60)
    print("COLLECTION COMPLETE")
    print("="*60)
    print(f"Total samples collected: {len(all_samples)}")
    
    # Count by domain
    domain_counts = {}
    for sample in all_samples:
        domain = sample['domain'].split('_')[0]  # Get main category
        domain_counts[domain] = domain_counts.get(domain, 0) + 1
    
    print("\nSamples per category:")
    for domain, count in sorted(domain_counts.items()):
        print(f"  {domain:20s}: {count:4d}")
    
    return all_samples


In [18]:

# Collect datasets
samples_per_domain = 200
eval_data = collect_datasets(samples_per_domain)



COLLECTING EVALUATION DATASETS

[1/17] MATH REASONING
------------------------------------------------------------
    Loading dataset... 1319 total examples. Sampling 200 examples... Done!
  ✓ GSM8K: 200 samples
  ✗ MATH failed: Dataset 'hendrycks/math' doesn't exist on the Hub or cannot be accessed.

[2/17] CODE GENERATION
------------------------------------------------------------
  ✓ HumanEval: 164 samples
    Loading MBPP... Done! 200 samples

[3/17] SCIENCE REASONING
------------------------------------------------------------
    Loading GPQA Diamond... Done! 198 samples
    Loading ARC Challenge... Done! 200 samples

[4/17] FACTUAL QA
------------------------------------------------------------
    Streaming 200 samples from mandarjoshi/trivia_qa... 10...20...30...40...50...60...70...80...90...100...110...120...130...140...150...160...170...180...190...200... Done!
  ✓ TriviaQA: 200 samples
    Streaming 600 samples from google-research-datasets/natural_questions... 10...20...

In [16]:
eval_df=pd.DataFrame(eval_data)
eval_df

,domain,question,answer,source
0,math_gsm8k,The girls are trying to raise money for a carn...,2280,math_gsm8k
1,math_gsm8k,Kalinda is working on a 360 piece puzzle with ...,1,math_gsm8k
2,math_gsm8k,Tom's ship can travel at 10 miles per hour. H...,5,math_gsm8k
3,math_gsm8k,James decides to buy birthday candles for his ...,12,math_gsm8k
4,math_gsm8k,Mariah’s grandma was teaching her to knit. Mar...,273,math_gsm8k
...,...,...,...,...
945,conversation,I'm feeling overwhelmed with work deadlines. W...,[Conversational - subjective],conversation
946,conversation,How should I prepare for my first job interview?,[Conversational - subjective],conversation
947,conversation,What are some good habits for maintaining work...,[Conversational - subjective],conversation
948,conversation,I want to learn a new programming language. Wh...,[Conversational - subjective],conversation


In [ ]:
# TriviaQA
try:
    triviaqa = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation",streaming=True)
    xdh = sample_hf_data(triviaqa)

    samples = []
    for item in list(xdh)[:samples_per_domain]:
        samples.append({
            "domain": "qa_trivia",
            "question": item["question"],
            "answer": item["answer"]["value"],
            "source": "qa_trivia"
        })
    all_samples.extend(samples)
    print(f"  ✓ TriviaQA: {len(samples)} samples")
except Exception as e:
    print(f"  ✗ TriviaQA failed: {e}")


  ✗ TriviaQA failed: Dataset.shuffle() got an unexpected keyword argument 'buffer_size'
